In [25]:
import os
import math
import numpy as np
import pandas as pd
# Constant for number of faces. This won't change unless dramatic changes are made to GCHP.
NUM_FACES = 6

In [26]:
def autoupdate_nx_ny(num_nodes, num_cores_per_node):
    """Automatically calculate NX and NY based on total cores."""
    total_cores = num_nodes * num_cores_per_node
    Z = total_cores // NUM_FACES
    N = math.ceil(math.sqrt(Z))

    while N > 0:
        if Z % N == 0:
            NX = N
            NY = Z // N
            return NX, NY
        N -= 1

    raise ValueError("Failed to compute NX and NY")

In [27]:
def verify_nx_ny(nx, ny, total_cores, cs_res):
    """Validate NX and NY settings based on total cores and resolution."""
    errors = []
    warnings = []

    if nx * ny * NUM_FACES != total_cores:
        errors.append("ERROR: NX*NY*NUM_FACES does not match total cores.")

    if cs_res % 2 != 0:
        errors.append("ERROR: CS_RES must be an even number.")

    if cs_res // nx < 4 or cs_res // ny < 4:
        errors.append("ERROR: Subdomain size too small. Each must be ≥ 4.")

    if nx // ny * 2 >= 5 or (ny // nx) * 2 >= 5:
        warnings.append("WARNING: NX x NY has a side ratio ≥ 2.5. Consider balancing.")

    return errors, warnings

In [28]:
NUM_CORES = 144
NUM_NODES = 2
NUM_CORES_PER_NODE = NUM_CORES // NUM_NODES
CS_RES = 90

NX, NY = autoupdate_nx_ny(NUM_NODES, NUM_CORES_PER_NODE)
errors, warnings = verify_nx_ny(NX, NY, NUM_NODES * NUM_CORES_PER_NODE, CS_RES)

print(f"NX = {NX}, NY = {NY}")
for e in errors:
    print(e)
for w in warnings:
    print(w)

NX = 4, NY = 6


In [34]:
def reshape_og_assignment(df: pd.DataFrame):
    """Restructure the DataFrame into NUM_FACES * RES * RES shape."""
    return df.values.reshape((NUM_FACES, CS_RES, CS_RES))

df = pd.read_csv(f"test/og_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv", index_col=0)
reshaped = reshape_og_assignment(df)
print("First face of reshaped data:")
# Use numpy's set_printoptions to fully print the array
print(reshaped[0])  # Print the first face for verification

First face of reshaped data:
[[ 0  0  0 ...  3  3  3]
 [ 0  0  0 ...  3  3  3]
 [ 0  0  0 ...  3  3  3]
 ...
 [20 20 20 ... 23 23 23]
 [20 20 20 ... 23 23 23]
 [20 20 20 ... 23 23 23]]


In [33]:
def nx_ny_to_reshaped_assignment(nx, ny):
    """Generate reshaped assignment based on NX and NY."""
    reshaped = np.zeros((NUM_FACES, CS_RES, CS_RES), dtype=int)
    core_id = 0
    for face in range(NUM_FACES):
        for i in range(ny):
            for j in range(nx):
                x_start = j * (CS_RES // nx)
                x_end = (j + 1) * (CS_RES // nx) if j < nx - 1 else CS_RES
                y_start = i * (CS_RES // ny)
                y_end = (i + 1) * (CS_RES // ny) if i < ny - 1 else CS_RES
                reshaped[face, y_start:y_end, x_start:x_end] = core_id
                core_id += 1
    return reshaped

generated_reshaped = nx_ny_to_reshaped_assignment(NX, NY)
print("First face of generated reshaped assignment:")
print(generated_reshaped[0])  # Print the first face for verification

First face of generated reshaped assignment:
[[ 0  0  0 ...  3  3  3]
 [ 0  0  0 ...  3  3  3]
 [ 0  0  0 ...  3  3  3]
 ...
 [20 20 20 ... 23 23 23]
 [20 20 20 ... 23 23 23]
 [20 20 20 ... 23 23 23]]


In [39]:
def compare_assignments(reshaped, generated):
    """Compare the original reshaped assignment with the generated one."""
    for face in range(NUM_FACES):
        if not np.array_equal(reshaped[face], generated[face]):
            print(f"Mismatch found in face {face}")
            # Find the first mismatch
            mismatch_indices = np.where(reshaped[face] != generated[face])
            print(f"Mismatch at indices: {list(zip(*mismatch_indices))}")
            return False
    print("All faces match.")
    return True


# Compare the original reshaped assignment with the generated one
comparison_result = compare_assignments(reshaped, generated_reshaped)

Mismatch found in face 0
Mismatch at indices: [(np.int64(0), np.int64(22)), (np.int64(0), np.int64(44)), (np.int64(0), np.int64(66)), (np.int64(1), np.int64(22)), (np.int64(1), np.int64(44)), (np.int64(1), np.int64(66)), (np.int64(2), np.int64(22)), (np.int64(2), np.int64(44)), (np.int64(2), np.int64(66)), (np.int64(3), np.int64(22)), (np.int64(3), np.int64(44)), (np.int64(3), np.int64(66)), (np.int64(4), np.int64(22)), (np.int64(4), np.int64(44)), (np.int64(4), np.int64(66)), (np.int64(5), np.int64(22)), (np.int64(5), np.int64(44)), (np.int64(5), np.int64(66)), (np.int64(6), np.int64(22)), (np.int64(6), np.int64(44)), (np.int64(6), np.int64(66)), (np.int64(7), np.int64(22)), (np.int64(7), np.int64(44)), (np.int64(7), np.int64(66)), (np.int64(8), np.int64(22)), (np.int64(8), np.int64(44)), (np.int64(8), np.int64(66)), (np.int64(9), np.int64(22)), (np.int64(9), np.int64(44)), (np.int64(9), np.int64(66)), (np.int64(10), np.int64(22)), (np.int64(10), np.int64(44)), (np.int64(10), np.int64

In [37]:
def reshape_generated_assignment(generated):
    """Reshape the generated assignment back to original format."""
    reshaped_df = pd.DataFrame(generated.reshape(NUM_FACES * CS_RES * CS_RES))
    reshaped_df.index.name = 'Column'
    reshaped_df.columns = ['Rank']
    return reshaped_df
reshaped_df = reshape_generated_assignment(generated_reshaped)


# Compare the reshaped DataFrame with the original one if it exists
def verify_and_export(
    reshaped_df,
    generated_path,
    original_path = None
):
    """Verify reshaped DataFrame against the original and export if necessary."""
    if os.path.exists(original_path):
        original_df = pd.read_csv(original_path, index_col=0)
        if not np.array_equal(original_df.values, reshaped_df.values):
            print("Mismatch found in reshaped DataFrame.")
            return False
        else:
            print("Reshaped DataFrame matches the original one.")
    else:
        print("Original file not found. Exporting reshaped DataFrame.")

    # Export if
    os.makedirs(os.path.dirname(generated_path), exist_ok=True)
    reshaped_df.to_csv(generated_path)
    return True

# Call the function with appropriate paths
verify_and_export(
    reshaped_df,
    f"test/generated_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv",
    f"test/og_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv"
)

Mismatch found in reshaped DataFrame.


False